In [1]:
!pip -q install langchain \
langchain-core \
langchain-community \
langchain-groq \
langchain-tavily \
langchain-experimental \
wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


# Task 1 – Understanding Tools

### 1. What is a tool in an AI Agent?

A tool is a function or external capability that an AI agent can use to perform tasks beyond text generation. Examples include calculators, search engines, databases, APIs, and document retrieval systems.

### 2. Why do agents need tools?

Agents need tools to access real-time information, perform accurate calculations, retrieve data, and interact with external systems. Tools make agents more capable than standalone language models.

### 3. Difference between a Chatbot and an Agent

Chatbot                                        AI Agent 

 Answers questions using an LLM         Uses an LLM plus external tools
Mainly conversational                        Can perform actions
Limited reasoning                       Can decide which tool to use
Cannot interact with external systems   Can search, calculate, retrieve data, etc

In [41]:
!pip -q install -U langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 5.6 MB/s eta 0:00:00


In [3]:
#importing all the libraries in this cell
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.tools import Tool
from langchain_core.tools import tool

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from datetime import datetime

/tmp/ipykernel_1770/62856722.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [45]:
#loading the groq api key from the google colab secrets
groq_api_key = userdata.get("GROQ_API_KEY")
llm = ChatGroq(model="llama-3.1-8b-instant",api_key=groq_api_key,temperature=0)
print("Groq LLM initialized successfully!")

Groq LLM initialized successfully!


In [6]:
#this cell is the codeblock of the calculator tool
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """
    Performs basic mathematical calculations.
    """
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"
print(calculator.invoke("25*18"))
print(calculator.invoke("25+18"))
print(calculator.invoke("25-18"))
print(calculator.invoke("25/18"))
print(calculator.invoke("25+18-908"))


450
43
7
1.3888888888888888
-865


In [8]:
#this is a wekipedia tool

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
import json

wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
try:
    result = wiki.invoke("Artificial Intelligence")
    print(result[:500])
except json.JSONDecodeError as e:
    print(f"Error decoding JSON from Wikipedia API: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Page: Artificial intelligence
Summary: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chan


In [ ]:
!pip install wikipedia
from wikipedia import summary
print(summary("Python", sentences=3))

In [9]:
#company tool policy
@tool
def company_policy_lookup(query: str) -> str:
    """
    Returns company policy information.
    """
    policies = {
        "leave": "Employees receive 20 paid leave days annually",
        "work": "Employees work Monday to Friday from 9 AM to 6 PM",
        "remote": "Employees can work remotely two days per week"}

    return policies.get(query.lower(),"Policy not found")
print(company_policy_lookup.invoke("leave"))

Employees receive 20 paid leave days annually


In [10]:
#creating more custon tools
@tool
def company_policy_lookup(query: str) -> str:
    """
    Returns company policy information.
    """
    policies = {
        "leave": "Employees receive 20 paid leave days annually",
        "work": "Employees work Monday to Friday from 9 AM to 6 PM",
        "remote": "Employees can work remotely two days per week"}
    return policies.get(query.lower(),"Policy not found.")
print(company_policy_lookup.invoke("leave"))

Employees receive 20 paid leave days annually


In [24]:
from datetime import datetime

@tool
def employee_database(name: str) -> str:
    """
    Returns employee designation.
    """
    database = {"dev": "AI Engineer","anand": "Data Scientist","jd": "Software Developer"}
    return database.get(name.lower(), "Employee not found.")
@tool
def current_time(dummy: str) -> str:
    """
    Returns the current date and time.
    """
    return datetime.now().strftime("%d-%m-%Y %H:%M:%S")

In [25]:
#creating the toolkit
tools = [calculator,wiki,company_policy_lookup,employee_database,current_time]

print("Toolkit created successfully!")
print(f"Total tools: {len(tools)}")

Toolkit created successfully!
Total tools: 5


Tool Calling Flow

User Query
        ↓
LLM analyzes the question
        ↓
Selects appropriate tool
        ↓
Executes the tool
        ↓
Observes tool output
        ↓
Generates final answer

In [26]:
#testing all the tools in one single cell
import json

print("Calculator:")
print(calculator.invoke("125+75"))

print("\nWikipedia:")
try:
    wiki_result = wiki.invoke("Python programming language")
    print(wiki_result[:300])
except json.JSONDecodeError as e:
    print(f"Error decoding JSON from Wikipedia API: {e}. The API might have returned a non-JSON response.")
except Exception as e:
    print(f"An unexpected error occurred with Wikipedia: {e}")

print("\nCompany Policy:")
print(company_policy_lookup.invoke("remote"))

print("\nEmployee Database:")
print(employee_database.invoke("jd"))

print("\nCurrent Time:")
print(current_time.invoke(""))

Calculator:
200

Wikipedia:
An unexpected error occurred with Wikipedia: Expecting value: line 1 column 1 (char 0)

Company Policy:
Employees can work remotely two days per week

Employee Database:
Software Developer

Current Time:
13-07-2026 12:27:21


In [27]:
#Binding the Tools to the Groq LLM

llm_with_tools = llm.bind_tools(tools)
print("Tools successfully bound to the LLM!")

Tools successfully bound to the LLM!


Wikipedia
    ↓
Calculator
    ↓
Final Answer

In [42]:
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(model=llm,tools=tools)
print("ReAct Agent created successfully!")

ReAct Agent created successfully!


/tmp/ipykernel_1770/1220924093.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=llm,tools=tools)


In [50]:
#Testing all the Agents
response = agent.invoke({"messages": [("user", "What is 125 * 48?")]})
print(response["messages"][-1].content)

The result of 125 * 48 is 6000.


In [ ]:
response = agent.invoke({"messages": [("user", "python")]})
print(response["messages"][-1].content)

ReAct Agent

ReAct stands for Reason + Act
The agent first reasons about the problem, selects an appropriate tool, observes the result, and then generates the final answer



In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_messages([
("system","You are an AI Agent. Think carefully before answering. Use tools whenever required."),
("human","{input}")])

Question
    ↓
Think
    ↓
Act (Tool)
    ↓
Observe
    ↓
Final Answer

task 11:
Observations

Benefits

- Uses external tools
- More accurate
- Can perform actions
- Solves complex tasks

Challenges

- Tool failures
- Higher latency
- More API calls
- Increased complexity

Chains vs Agents

Chains follow a fixed sequence.
Agents dynamically decide which tools to use.

Agents vs RAG
Use RAG for document-based question answering
Use Agents when multiple tools and decision-making are required